# Õppetund 05 - Agentne RAG


## Seadistamine

See märkmik demonstreerib Agentic RAG (päringutega täiustatud generatsiooni) mustrit, kasutades Microsoft Agent Frameworki.

**Eeltingimused:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — teie Azure AI Search teenuse lõpp-punkt
- `AZURE_SEARCH_API_KEY` — teie Azure AI Search API võti
- Azure OpenAI keskkonnamuutujate kaudu seadistatud juurutus
- Azure CLI autentitud (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mis on Agentic RAG?

Traditsiooniline RAG järgib fikseeritud torujuhtme: esmalt tuuakse dokumendid, seejärel genereeritakse vastus. **Agentic RAG** läheb kaugemale, andes agendile autonoomia otsustada, **millal** ja **kuidas** teavet otsida.

Agentic RAG-iga saab agent:
- **Otsustada**, kas enne küsimusele vastamist on vaja otsida
- **Valida**, millist andmeallikat või tööriista pärida
- **Hinnata** otsingutulemusi ja vajadusel teha järeleotsinguid, kui esimene katse ei ole piisav
- **Kombineerida** teavet mitmest otsinguetapist kooskõlastatud vastuseks

See teeb agendi paindlikumaks ja täpsemaks võrreldes staatilise otsi-siis-generaator torujuhtmega.


## Otsingutööriista loomine

Agentic RAG-s on välised andmeallikad pakitud **tööriistadeks**, mida agent saab vajadusel kutsuda. See võimaldab agendil käsitleda päringut lihtsalt ühe tegevusena, mida ta saab teha, mitte kohustusliku sammuna.

Allpool määratleme reisiteabe baasi ja muudame selle tööriistaks, mida agent saab kutsuda sihtkoha teabe otsimiseks.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG-agendi loomine

Nüüd loome agendi, kellele on antud juhis **alati otsida infot enne vastamist**. Agent kasutab tööriista `search_travel_knowledge`, et oma vastuseid teadmistebaasil kinnistada, selle asemel et tugineda ainult oma koolitusandmetele.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iteratiivne päring — Maker-Checker mustrit

Agentic RAG peamine eelis on **iteratiivne päring**. Agent saab teha mitu otsinguringi, et kontrollida, täiendada või laiendada oma algseid leide — sarnaselt "maker-checker" töövoogule:

1. **Maker-ülesanne**: Agent hangib esmased andmed ja koostab vastuse eelnõu.
2. **Checker-ülesanne**: Agent teeb täiendavaid päringuid, et kontrollida detaile või täita tühimikud.

Allpool esitatakse agendile küsimus, mis nõuab mitme sihtkoha võrdlemist, mis ajendab teda otsima mitu korda.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Kokkuvõte

Selles õppetükis õppisite, kuidas luua **Agentic RAG** süsteem Microsofti Agendi Raamistiku abil:

- **Agentic RAG** võimaldab agentidel iseseisvalt otsustada, millal infot hankida, muutes informatsiooni hankimise dünaamiliseks, mitte fikseeritud.
- **Tööriistad kui andmeallikad**: Välised teadmistebaasid (näiteks Azure AI Search) on pakitud tööriistadeks, mida agent saab kutsuda.
- **Iteratiivne hankimine**: maker-checker mustrit kasutades saab agent teha mitu hankimiskorda — otsida, kontrollida ja täiustada — enne lõpliku vastuse andmist.

Tootmiskeskkonnas asendaksite mälupealse `TRAVEL_KNOWLEDGE_BASE` reaalse Azure AI Search indeksiga, et hallata suuremahulist reisimisdokumentide hankimist.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastutusest loobumine**:  
See dokument on tõlgitud kasutades tehisintellektil põhinevat tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüame täpsust, palun pidage meeles, et automaatsed tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks lugeda autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta ühegi sellisest tõlkest tuleneva valesti mõistmise või tõlgendamise eest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
